In [2]:
"""
================================================================================
[Engram Architecture Demo Implementation]

DISCLAIMER:
1. Demo Purpose Only: 
   This code is a demonstration version intended to illustrate the core logic and 
   data flow of the Engram module.

2. Production Readiness: 
   This implementation requires further optimization for actual production use 
   (e.g., custom CUDA kernels, distributed training support).

3. Simplifications: 
   Standard components (Normalization, Attention, MoE) and complex Hyper-connection 
   mechanisms are omitted or mocked in this version to focus exclusively on the 
   Engram module implementation.
================================================================================
"""

"""
pip install torch numpy transformers sympy
"""

## built-in
from typing import List
from dataclasses import dataclass, field
import math

## third-party
from sympy import isprime
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer
from tokenizers import normalizers, Regex 


/Users/yingyao/miniconda3/envs/transformer-practice/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@dataclass
class EngramConfig:
    tokenizer_name_or_path: str = "/Users/yingyao/Desktop/Code.nosync/GetHandsDirty.nosync/sft/tokenizer"
    engram_vocab_size: List[int] = field(default_factory=lambda: [6400*5, 6400*5]) # vocab of tokenizer used
    max_ngram_size: int = 3
    n_embed_per_ngram: int = 512
    n_head_per_ngram: int = 8 # to mitigate hash collison
    layer_ids: List[int] = field(default_factory=lambda: [1, 15])
    pad_id: int = 2
    seed: int = 0
    kernel_size: int = 4
    
@dataclass
class BackBoneConfig:
    hidden_size: int = 1024
    hc_mult: int = 4
    vocab_size: int = 6400
    num_layers: int = 30
    
engram_cfg = EngramConfig()
backbone_config = BackBoneConfig()

In [4]:
class CompressedTokenizer:
    def __init__(
        self,
        tokenizer_name_or_path,
    ):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path, trust_remote_code=True)
        
        SENTINEL = "\uE000"

        # uniform capitals/small letters, remove special symbols, remove space etc.
        self.normalizer = normalizers.Sequence([
            normalizers.NFKC(),
            normalizers.NFD(),
            normalizers.StripAccents(),
            normalizers.Lowercase(),
            normalizers.Replace(Regex(r"[ \t\r\n]+"), " "),
            normalizers.Replace(Regex(r"^ $"), SENTINEL),
            normalizers.Strip(),
            normalizers.Replace(SENTINEL, " "),
        ])
        
        self.lookup_table, self.num_new_token = self._build_lookup_table()
    
    def __len__(self):
        return self.num_new_token
    
    # map original token id to compressed token id
    def _build_lookup_table(self):
        old2new = {}
        key2new = {}          
        new_tokens = []

        vocab_size = len(self.tokenizer)
        for tid in range(vocab_size):
            text = self.tokenizer.decode([tid], skip_special_tokens=False)
            
            if "�" in text:
                key = self.tokenizer.convert_ids_to_tokens(tid)
            else:
                norm = self.normalizer.normalize_str(text)
                key = norm if norm else text

            nid = key2new.get(key)
            if nid is None:
                nid = len(new_tokens)
                key2new[key] = nid
                new_tokens.append(key)
            old2new[tid] = nid # point original token id to new token id
        
        lookup = np.empty(vocab_size, dtype=np.int64) # create a new lookup table that have same size as original tokenizer
        for tid in range(vocab_size):
            lookup[tid] = old2new[tid] # new lookup's different tid may point to same nid

        return lookup, len(new_tokens)
    
    def _compress(self, input_ids):
        arr = np.asarray(input_ids, dtype=np.int64)
        pos_mask = arr >= 0
        out = arr.copy()
        valid_ids = arr[pos_mask]
        out[pos_mask] = self.lookup_table[valid_ids]
        return out    
    
    def __call__(self, input_ids):
        return self._compress(input_ids)
        

In [5]:
tokenizer = AutoTokenizer.from_pretrained(engram_cfg.tokenizer_name_or_path,trust_remote_code=True)
print('Original vocab size:', tokenizer.vocab_size)

text = 'Hi, world! Hi, World!'
input_ids = tokenizer.encode(text)
print('Original input ids:', input_ids)

compressed_tokenizer = CompressedTokenizer(engram_cfg.tokenizer_name_or_path)
compressed_input_ids = compressed_tokenizer(input_ids)
print('Compressed input ids:', compressed_input_ids)

Original vocab size: 6400
Original input ids: [561, 76, 15, 1637, 4, 561, 76, 15, 6318, 4]
Compressed input ids: [  43   44   15 1474    4   43   44   15 1474    4]


In [6]:
new_vocab_size = len(compressed_tokenizer)
1 - new_vocab_size / compressed_tokenizer.tokenizer.vocab_size

0.08859375000000003

In [7]:
class ShortConv(nn.Module):
    def __init__(
        self, 
        hidden_size: int, 
        kernel_size: int = 4, 
        dilation: int = 1, 
        norm_eps: float = 1e-5,
        hc_mult: int = 4,
        activation: bool = True,
    ):
        super().__init__()
        self.hc_mult = hc_mult
        self.activation = activation
        
        total_channels = hidden_size * hc_mult
        self.conv = nn.Conv1d(
            in_channels=total_channels,
            out_channels=total_channels,
            kernel_size=kernel_size,
            groups=total_channels,
            bias=False,
            padding=(kernel_size - 1) * dilation,
            dilation=dilation,
        )

        self.norms = nn.ModuleList([
            nn.RMSNorm(hidden_size, eps=norm_eps) 
            for _ in range(hc_mult)
        ])
        
        if self.activation:
            self.act_fn = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Input:  (B,T,HC_MULT,D)
        Output: (B,T,HC_MULT,D)
        """
        B, T, G, C = x.shape
        
        assert G == self.hc_mult, f"Input groups {G} != hc_mult {self.hc_mult}"

        normed_chunks = []
        for i in range(G):
            chunk = x[:, :, i, :]
            normed_chunks.append(self.norms[i](chunk))
        
        x_norm = torch.cat(normed_chunks, dim=-1) # (B, T, G, C)
        x_bct = x_norm.transpose(1, 2) # (B, G, T, C)    
        y_bct = self.conv(x_bct)
        y_bct = y_bct[..., :T]

        if self.activation:
            y_bct = self.act_fn(y_bct)
        y = y_bct.transpose(1, 2).view(B, T, G, C).contiguous() # (B, G, T, C) -> (B, T, G, C)  
        
        return y

In [8]:
def find_next_prime(start, seen_primes):
    candidate = start + 1
    while True:
        if isprime(candidate) and candidate not in seen_primes:
            return candidate
        candidate += 1        

In [9]:
find_next_prime(1, [1, 2, 3])

5

In [ ]:
class NgramHashMapping:
    def __init__(
        self, 
        engram_vocab_size,
        max_ngram_size,
        n_embed_per_ngram,
        n_head_per_ngram,
        layer_ids,
        tokenizer_name_or_path,
        pad_id,
        seed,  
    ):
        self.vocab_size_per_ngram = engram_vocab_size
        self.max_ngram_size = max_ngram_size
        self.n_embed_per_ngram = n_embed_per_ngram
        self.n_head_per_ngram = n_head_per_ngram
        self.pad_id = pad_id
        self.layer_ids = layer_ids

        self.compressed_tokenizer = CompressedTokenizer(
            tokenizer_name_or_path=tokenizer_name_or_path
        )            
        self.tokenizer_vocab_size = len(self.compressed_tokenizer)
        if self.pad_id is not None:
            self.pad_id = int(self.compressed_tokenizer.lookup_table[self.pad_id])

        max_long = np.iinfo(np.int64).max
        M_max = int(max_long // self.tokenizer_vocab_size)
        # to ensure the integer does not cause an integer overflow during hashing process
        half_bound = max(1, M_max // 2) 
        PRIME_1 = 10007
        
        self.layer_multipliers = {}

        for layer_id in self.layer_ids:
            base_seed = int(seed + PRIME_1 * int(layer_id))
            g = np.random.default_rng(base_seed)
            # generate a 1D array that have max_ngram_size random integers
            r = g.integers(
                low=0,
                high=half_bound,
                size=(self.max_ngram_size,),
                dtype=np.int64
            ) 
            # therefore, the multiplier * token_id does not cause integer overflow
            multipliers = r * 2 + 1
            self.layer_multipliers[layer_id] = multipliers

        self.vocab_size_across_layers = self.calculate_vocab_size_across_layers()

    def calculate_vocab_size_across_layers(self):
        seen_primes = set()
        vocab_size_across_layers = {}
        
        for layer_id in self.layer_ids: 
            all_ngram_vocab_sizes = []
            
            # for each layer, for each n-gram from 2 to max_ngram_size, create its vocab size for all heads 
            # that using prime numbers which are bigger than original vocab size
            for ngram in range(2, self.max_ngram_size + 1):
                current_ngram_heads_sizes = []
                
                vocab_size = self.vocab_size_per_ngram[ngram - 2] # 6400
                num_head = self.n_head_per_ngram # 8
                current_prime_search_start = vocab_size - 1
                
                for _ in range(num_head):
                    found_prime = find_next_prime(
                        current_prime_search_start, 
                        seen_primes
                    )
                    seen_primes.add(found_prime)
                    current_ngram_heads_sizes.append(found_prime)
                    current_prime_search_start = found_prime
                
                all_ngram_vocab_sizes.append(current_ngram_heads_sizes)
            vocab_size_across_layers[layer_id] = all_ngram_vocab_sizes # shape: (max_ngram_size - 1, n_head_per_ngram)
            
        return vocab_size_across_layers

    def _get_ngram_hashes(
        self,
        input_ids: np.ndarray,
        layer_id: int,
    ) -> np.ndarray:
        x = np.asarray(input_ids, dtype=np.int64)
        B, T = x.shape

        multipliers = self.layer_multipliers[layer_id]

        def shift_k(k: int) -> np.ndarray:
            if k == 0: return x
            shifted = np.pad(x, ((0, 0), (k, 0)),
                                mode='constant', constant_values=self.pad_id)[:, :T]
            return shifted

        base_shifts = [shift_k(k) for k in range(self.max_ngram_size)] # [[tid0, tid1, ...], [pid, tid0, tid1, ...], [pid, pid, tid0, tid1, ...]]

        all_hashes = []
        
        for n in range(2, self.max_ngram_size + 1):
            n_gram_index = n - 2
            tokens = base_shifts[:n]
            mix = (tokens[0] * multipliers[0])
            for k in range(1, n):
                # get a larger int to represent these tokens' k-gram by Bitwise XOR, e.g. mix = [[mix_tid0, mix_tid1, .., mix_tidT]]
                mix = np.bitwise_xor(mix, tokens[k] * multipliers[k])
            num_heads_for_this_ngram = self.n_head_per_ngram
            head_vocab_sizes = self.vocab_size_across_layers[layer_id][n_gram_index]
            
            # to use the larger int as index of the embedding table, we need to shrink it using mod,
            # but then it may occur hasing collision issue, e.g. vocab_size: 100, "apple pie" -> 2-gram mix: 500, 
            # "car tire" -> 2-gram mix: 1000, then both hash = mix % mod = 0
            # to mitigate this issue, using multi-heads, then the prob that all head's hash values are the same is very low for different n-grams
            for j in range(num_heads_for_this_ngram):
                mod = int(head_vocab_sizes[j])
                # get the final hash value for this n-gram and take mod by the vocab size of this head 
                head_hash = mix % mod
                all_hashes.append(head_hash.astype(np.int64, copy=False)) 
        
        return np.stack(all_hashes, axis=2) # shape: (B, T, n_head_per_ngram * (max_ngram_size-1))

    def hash(self, input_ids):
        input_ids = self.compressed_tokenizer(input_ids)
        hash_ids_for_all_layers = {}
        for layer_id in self.layer_ids:
            hash_ids_for_all_layers[layer_id] = self._get_ngram_hashes(input_ids, layer_id=layer_id)
        return hash_ids_for_all_layers
    
hash_mapping = NgramHashMapping(
            engram_vocab_size=engram_cfg.engram_vocab_size,
            max_ngram_size = engram_cfg.max_ngram_size,
            n_embed_per_ngram = engram_cfg.n_embed_per_ngram,
            n_head_per_ngram = engram_cfg.n_head_per_ngram,
            layer_ids = engram_cfg.layer_ids,
            tokenizer_name_or_path=engram_cfg.tokenizer_name_or_path,
            pad_id = engram_cfg.pad_id,
            seed = engram_cfg.seed)

vocab_size_across_layers = hash_mapping.vocab_size_across_layers
print('Vocab size across layers and n-grams:', vocab_size_across_layers)
print(len(vocab_size_across_layers[1][0]))

print('___________')
input_ids_2 = [input_ids]  # convert to (1, T)
hash_ids_for_all_layers = hash_mapping.hash(input_ids_2)
print('Hash IDs for all layers:', hash_ids_for_all_layers)
print(hash_ids_for_all_layers[1].shape)

Vocab size across layers and n-grams: {1: [[32003, 32009, 32027, 32029, 32051, 32057, 32059, 32063], [32069, 32077, 32083, 32089, 32099, 32117, 32119, 32141]], 15: [[32143, 32159, 32173, 32183, 32189, 32191, 32203, 32213], [32233, 32237, 32251, 32257, 32261, 32297, 32299, 32303]]}
8
___________
Hash IDs for all layers: {1: array([[[18981, 24314,  5308, 26960, 26534, 13463,  7472, 17635, 20591,
         29040,  8895, 22866,   460, 18423,   623, 14286],
        [31869, 29809, 31281, 12866, 25849,  6878, 18438, 11352, 29541,
           458, 28939, 27911, 23822, 31415, 16541,  1621],
        [11122, 15746, 26088,  3706, 24318, 19836,   715, 27110, 16521,
          9263, 31651,  4486, 17044,  7759,  1777, 30223],
        [31099, 16639, 31169, 20418, 10594,  2592, 21937, 27262, 14085,
         20400, 27773,  4536, 26484, 27553, 25910,  1143],
        [25948,  1957,  6533,  8668, 10523, 30841,  6673,  5381,  3640,
         20284, 27488, 14344, 14854,   638,  3186, 11252],
        [ 9191, 1475

In [11]:
class MultiHeadEmbedding(nn.Module):
    def __init__(self, list_of_N: List[int], D: int):
        super().__init__()
        self.num_heads = len(list_of_N)
        self.embedding_dim = D
        
        # now, put the hash values for n-gram's n-heads into one table, so need to arrange the table's index
        offsets = [0]
        for n in list_of_N[:-1]:
            offsets.append(offsets[-1] + n)
        
        self.register_buffer("offsets", torch.tensor(offsets, dtype=torch.long))
        
        total_N = sum(list_of_N) # total size of the embedding table
        self.embedding = nn.Embedding(num_embeddings=total_N, embedding_dim=D) 

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # input_ids shape: (B, T, engram_hidden_size)
        shifted_input_ids = input_ids + self.offsets
        output = self.embedding(shifted_input_ids) # shape: (B, T, engram_hidden_size, D)
        
        return output

In [12]:
list_of_N = [x for y in hash_mapping.vocab_size_across_layers[1] for x in y]
print('list_of_N:', list_of_N)

list_of_N: [32003, 32009, 32027, 32029, 32051, 32057, 32059, 32063, 32069, 32077, 32083, 32089, 32099, 32117, 32119, 32141]


In [ ]:
multi_head_embedding = MultiHeadEmbedding(
        list_of_N = [x for y in hash_mapping.vocab_size_across_layers[1] for x in y],
        D = engram_cfg.n_embed_per_ngram // engram_cfg.n_head_per_ngram,
    )
hash_input_ids = torch.from_numpy(hash_mapping.hash(input_ids_2)[1])  # shape: (B, T, n_head_per_ngram * (max_ngram_size-1))
print('hash_input_ids shape:', hash_input_ids.shape)

multi_head_embedding(hash_input_ids).shape

hash_input_ids shape: torch.Size([1, 10, 16])


torch.Size([1, 10, 16, 64])

In [ ]:
class Engram(nn.Module):
    def __init__(self, layer_id):
        super().__init__()
        self.layer_id = layer_id
        self.hash_mapping = NgramHashMapping(
            engram_vocab_size=engram_cfg.engram_vocab_size,
            max_ngram_size = engram_cfg.max_ngram_size,
            n_embed_per_ngram = engram_cfg.n_embed_per_ngram,
            n_head_per_ngram = engram_cfg.n_head_per_ngram,
            layer_ids = engram_cfg.layer_ids,
            tokenizer_name_or_path=engram_cfg.tokenizer_name_or_path,
            pad_id = engram_cfg.pad_id,
            seed = engram_cfg.seed,
        )
        self.multi_head_embedding = MultiHeadEmbedding(
            # flatten n-gram vocab sizes for all heads of this layer, e.g. n_head_per_ngram * (max_ngram_size - 1) elements in the list
            list_of_N = [x for y in self.hash_mapping.vocab_size_across_layers[self.layer_id] for x in y], 
            D = engram_cfg.n_embed_per_ngram // engram_cfg.n_head_per_ngram,
        )
        self.short_conv = ShortConv(
            hidden_size = backbone_config.hidden_size,
            kernel_size = engram_cfg.kernel_size,
            dilation    = engram_cfg.max_ngram_size,
            hc_mult     = backbone_config.hc_mult,
        )
        engram_hidden_size = (engram_cfg.max_ngram_size-1) * engram_cfg.n_embed_per_ngram
        self.value_proj = nn.Linear(engram_hidden_size, backbone_config.hidden_size) # (B, T, hidden_size)
        self.key_projs = nn.ModuleList(
            [nn.Linear(engram_hidden_size, backbone_config.hidden_size) for _ in range(backbone_config.hc_mult)]
        )
        self.norm1 = nn.ModuleList([nn.RMSNorm(backbone_config.hidden_size) for _ in range(backbone_config.hc_mult)])
        self.norm2 = nn.ModuleList([nn.RMSNorm(backbone_config.hidden_size) for _ in range(backbone_config.hc_mult)])
    
    def forward(self,hidden_states,input_ids):
        """
        hidden_states: [B, T, HC_MULT, D]
        input_ids: [B, T]
        """
        hash_input_ids = torch.from_numpy(self.hash_mapping.hash(input_ids)[self.layer_id])
        embeddings = self.multi_head_embedding(hash_input_ids).flatten(start_dim=-2) # shape: (B, T, engram_hidden_size, D) -> (B, T, engram_hidden_size * D)
        gates = []
        for hc_idx in range(backbone_config.hc_mult):
            key = self.key_projs[hc_idx](embeddings) # shape: (B, T, hidden_size)
            normed_key = self.norm1[hc_idx](key) # normalize m-th branch's key
            query = hidden_states[:,:,hc_idx,:] 
            normed_query = self.norm2[hc_idx](query) # normalize m-th branch's query
            gate = (normed_key * normed_query).sum(dim=-1) / math.sqrt(backbone_config.hidden_size)
            gate = gate.abs().clamp_min(1e-6).sqrt() * gate.sign() # 
            gate = gate.sigmoid().unsqueeze(-1) # shape: (B, T, 1)
            gates.append(gate)
        gates = torch.stack(gates, dim=2) # shape: (B, T, HC_MULT, 1)
        value = gates * self.value_proj(embeddings).unsqueeze(2) # shape: (B, T, HC_MULT, hidden_size)
        output = value + self.short_conv(value) # shape: (B, T, HC_MULT, hidden_size)
        return output 

engram = Engram(layer_id=1)
hidden_states = torch.randn(1, 10, 4, 1024)
output = engram(hidden_states, input_ids_2)
print(output.shape)

torch.Size([1, 10, 4, 1024])


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self,layer_id):
        super().__init__()
        self.attn = lambda x:x
        self.moe  = lambda x:x
        self.engram = None
        if layer_id in engram_cfg.layer_ids:
            self.engram = Engram(layer_id=layer_id)
    
    def forward(self,input_ids,hidden_states):
        if self.engram is not None:
            hidden_states = self.engram(hidden_states=hidden_states,input_ids=input_ids) + hidden_states # can replace w mHC
        hidden_states = self.attn(hidden_states) + hidden_states 
        hidden_states = self.moe(hidden_states) + hidden_states  # can replace w mHC
        return hidden_states


In [24]:
LLM = [
    nn.Embedding(backbone_config.vocab_size,backbone_config.hidden_size),
    *[TransformerBlock(layer_id=layer_id) for layer_id in range(backbone_config.num_layers)],
    nn.Linear(backbone_config.hidden_size, backbone_config.vocab_size)
]

text = "Only Alexander the Great could tame the horse Bucephalus."
tokenizer = AutoTokenizer.from_pretrained(engram_cfg.tokenizer_name_or_path,trust_remote_code=True)
input_ids = tokenizer(text,return_tensors='pt').input_ids

B,L = input_ids.shape

for idx, layer in enumerate(LLM):
    if idx == 0:
        hidden_states = LLM[0](input_ids)
        ## mock hyper-connection
        hidden_states = hidden_states.unsqueeze(2).expand(-1, -1, backbone_config.hc_mult, -1)      
    elif idx == len(LLM)-1:
        ## mock hyper-connection
        hidden_states = hidden_states[:,:,0,:] 
        output = layer(hidden_states)
    else:
        hidden_states = layer(input_ids=input_ids,hidden_states=hidden_states)

print("✅ Forward Complete!")
print(f"{input_ids.shape=}\n{output.shape=}")

✅ Forward Complete!
input_ids.shape=torch.Size([1, 23])
output.shape=torch.Size([1, 23, 129280])
